# Act 3: a workflow and its harness

The model proposes a cart. Application code chooses the next step, validates the cart, obtains manager approval when required, and performs mock checkout.

Start with repeatable **offline fixtures**, then switch to a live proposing agent. No real purchase or email is sent. This notebook requires a Python 3.13+ kernel, like Acts 1–2; Colab runtime compatibility has not been validated here.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

root = Path.cwd()
if not (root / "formaggio" / "config.py").exists():
    if (root.parent / "formaggio" / "config.py").exists():
        root = root.parent
    else:
        root = root / "agent-harness"
        if not root.exists():
            subprocess.run(["git", "clone", "https://github.com/dlops-io/agent-harness.git", str(root)], check=True)
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
print("Repository:", root)

In [ ]:
import tomllib

if sys.version_info < (3, 13):
    raise RuntimeError("Select a Python 3.13+ notebook kernel before continuing.")
project = tomllib.loads(Path("pyproject.toml").read_text())
subprocess.run([sys.executable, "-m", "pip", "install", *project["project"]["dependencies"]], check=True)
subprocess.run([sys.executable, "-m", "scripts.check_sdk"], check=True)

## 1. Run the workflow

This fixture starts with an invalid stock proposal and then supplies a corrected cart. Read the reported steps and the authoritative outcome.


In [ ]:
from pathlib import Path
from acts.act3_workflow import build_act3, build_workflow
from cli import print_workflow_result
from formaggio.operations.observability import Recorder

DB_PATH = Path("outputs/act3_notebook.sqlite")
lesson = build_act3(fixture=True, scenario="out-of-stock")


In [ ]:
with Recorder(DB_PATH) as recorder:
    result = await lesson.run(recorder, progress=print)
print_workflow_result(result)


## 2. Read the graph

**Agent systems:** one model proposes; code controls transitions. **Context engineering:** each revision receives the original request, current shop context, previous proposal, and validation feedback.

Follow `validate → propose` for an invalid cart and `validate → pairings` for a valid cart. Open `formaggio/agents/workflow_steps.py` to inspect the node implementations.


In [ ]:
import inspect
print(inspect.getsource(build_workflow))


## 3. Add or remove named execution features

**Agent harness:** `build_act3` adds `WorkflowTrace()` and `WorkflowBudget(model_calls=8, seconds=120)`. The timer counts active execution across resumes; manager wait time is excluded. Each `.run(...)` uses fresh runtime state.

Removing trace changes observability. Validation, authorization, manager review, final stock checks, and mandatory audit remain in the graph and checkout code.


In [ ]:
quiet_lesson = lesson.without("trace")
with Recorder(DB_PATH) as recorder:
    result_without_trace = await quiet_lesson.run(recorder, progress=print)
print_workflow_result(result_without_trace)


## 4. Pause for manager review

The fixture below totals $220.80, so approval is required. This callback deliberately supplies a **fixture decision**. Change `approve` to `decline` and rerun to compare outcomes. Approval does not override stock, allergy, or other checkout rules.


In [ ]:
async def fixture_manager(ticket):
    print(f"Reviewing fixture cart: ${ticket.report.subtotal_cents / 100:.2f}")
    return "approve"  # An explicit classroom decision, not a model tool.

approval_lesson = build_act3(fixture=True, scenario="manager-approval", decision_source="fixture")
with Recorder(DB_PATH) as recorder:
    approval_result = await approval_lesson.run(recorder, manager=fixture_manager, progress=print)
print_workflow_result(approval_result)


## 5. Use the live proposing agent (optional)

These cells need your OpenAI credentials and make model API calls. The manager callback is host code, outside the model's tools. It prompts only when the validated cart requires approval. Reuse the supported runtime and model configuration from Acts 1–2.


In [ ]:
import getpass
import os
from formaggio.config import MODEL

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")


In [ ]:
async def review_cart(ticket):
    print(f"Manager review: ${ticket.report.subtotal_cents / 100:.2f}")
    print(ticket.request)
    print(ticket.report.items)
    while True:
        decision = input("Approve or decline? ").strip().lower()
        if decision in {"approve", "decline"}:
            return decision

live_lesson = build_act3(model=MODEL, scenario="manager-approval")
with Recorder(DB_PATH) as recorder:
    live_result = await live_lesson.run(recorder, manager=review_cart, progress=print)
print_workflow_result(live_result)


## Reflect

- Which actions came from the proposing agent, and which came from application code?
- Why does checkout revalidate after manager review?
- Which observations disappear without trace, and which checks remain?
- **Evaluation:** inspect `workflow.result`, `cart.validated`, and `approval.decided` events. Fixture success verifies a known path; it does not measure live proposal quality.

Skills, longer-running planner state, and composition arrive in later acts.
